# Video Telemetry QoS Pipeline - Testing Suite

This notebook provides automated testing for the video telemetry analytics pipeline to ensure production readiness.

## Test Categories

1. **Schema Validation Tests** - Verify table schemas match expected contracts
2. **Data Quality Tests** - Validate data contract expectations and constraints
3. **Data Flow Tests** - Ensure data flows correctly through Bronze → Silver → Gold
4. **Quarantine Logic Tests** - Verify quality checks catch bad data
5. **Aggregation Tests** - Validate gold layer metric calculations
6. **Performance Tests** - Check query execution times
7. **Pipeline Health Tests** - Verify pipeline configuration and status

## Usage

**Run all cells** to execute the full test suite. Each test will:
- ✅ **PASS** if the test succeeds
- ❌ **FAIL** if the test fails with error details
- ⚠️ **WARN** if the test passes with warnings

## Test Results Summary

After running all tests, check the final summary cell for:
- Total tests run
- Passed/Failed/Warning counts
- List of failed tests requiring attention

In [0]:
# Test framework setup
import datetime
from pyspark.sql import functions as F
from typing import Dict, List, Tuple

# Pipeline configuration
CATALOG = "workspace"
SCHEMA = "hive_video_analytics"
PIPELINE_ID = "4296766b-887a-4ab4-9eef-2e6e07a32b66"

# Table names
TABLES = {
    "bronze": f"{CATALOG}.{SCHEMA}.bronze_telemetry_raw",
    "silver": f"{CATALOG}.{SCHEMA}.silver_telemetry_enriched",
    "quarantine": f"{CATALOG}.{SCHEMA}.silver_telemetry_quarantine",
    "gold": f"{CATALOG}.{SCHEMA}.gold_viewer_qos_metrics"  # Fixed: materialized view name
}

# Test results tracking
test_results = []

def run_test(test_name: str, test_func, category: str = "General"):
    """
    Execute a test function and track results.
    
    Args:
        test_name: Descriptive name of the test
        test_func: Function that returns (passed: bool, message: str, warning: bool)
        category: Test category for grouping
    """
    print(f"\n{'='*80}")
    print(f"Running: {test_name}")
    print(f"Category: {category}")
    print(f"{'='*80}")
    
    try:
        passed, message, is_warning = test_func()
        
        if passed and not is_warning:
            status = "✅ PASS"
            print(f"\n{status}: {message}")
        elif passed and is_warning:
            status = "⚠️ WARN"
            print(f"\n{status}: {message}")
        else:
            status = "❌ FAIL"
            print(f"\n{status}: {message}")
        
        test_results.append({
            "category": category,
            "test_name": test_name,
            "status": status,
            "message": message,
            "timestamp": datetime.datetime.now()
        })
        
    except Exception as e:
        status = "❌ FAIL"
        error_msg = f"Test threw exception: {str(e)}"
        print(f"\n{status}: {error_msg}")
        
        test_results.append({
            "category": category,
            "test_name": test_name,
            "status": status,
            "message": error_msg,
            "timestamp": datetime.datetime.now()
        })

print("✅ Test framework initialized")
print(f"Pipeline ID: {PIPELINE_ID}")
print(f"Testing tables: {list(TABLES.keys())}")
print(f"\nNote: Gold layer is a MATERIALIZED VIEW (gold_viewer_qos_metrics), not a streaming table")

In [0]:
# Test 1: Verify all tables exist
def test_table_existence():
    """
    Verify that all expected pipeline tables exist in the catalog.
    """
    missing_tables = []
    
    for table_type, table_name in TABLES.items():
        try:
            # Try to read table metadata
            df = spark.sql(f"DESCRIBE TABLE {table_name}")
            row_count = df.count()
            print(f"  ✓ Found {table_type}: {table_name} ({row_count} columns)")
        except Exception as e:
            print(f"  ✗ Missing {table_type}: {table_name}")
            missing_tables.append(table_name)
    
    if missing_tables:
        return False, f"Missing tables: {', '.join(missing_tables)}", False
    else:
        return True, f"All {len(TABLES)} tables exist", False

run_test("Table Existence Check", test_table_existence, "Schema Validation")

In [0]:
# Test 2: Validate bronze layer schema
def test_bronze_schema():
    """
    Verify bronze_telemetry_raw has the expected schema structure.
    """
    expected_columns = [
        "customerId", "contentId", "clientId", "timestampInfo",
        "player", "totalDistribution", "qualityDistribution", "eventDate"
    ]
    
    df = spark.table(TABLES["bronze"])
    actual_columns = df.columns
    
    missing = [col for col in expected_columns if col not in actual_columns]
    extra = [col for col in actual_columns if col not in expected_columns and col != "_rescued_data"]
    
    issues = []
    if missing:
        issues.append(f"Missing columns: {missing}")
    if extra:
        issues.append(f"Unexpected columns: {extra}")
    
    # Check nested structure
    schema = df.schema
    has_timestamp_server = "timestampInfo.server" in [f"{field.name}.server" for field in schema.fields if field.name == "timestampInfo"]
    
    if not has_timestamp_server:
        # Verify nested structure exists
        timestamp_field = [f for f in schema.fields if f.name == "timestampInfo"]
        if timestamp_field:
            print(f"  ✓ timestampInfo structure: {timestamp_field[0].dataType}")
    
    if issues:
        return False, "; ".join(issues), False
    else:
        return True, f"Bronze schema valid with {len(actual_columns)} columns", False

run_test("Bronze Layer Schema Validation", test_bronze_schema, "Schema Validation")

In [0]:
# Test 3: Validate silver layer schema
def test_silver_schema():
    """
    Verify silver_telemetry_enriched has the expected flattened schema.
    """
    expected_columns = [
        "customerId", "contentId", "clientId",
        "timestamp_server", "timestamp_agent",
        "buffering_count", "buffering_time_ms", "buffering_time_sec",
        "quality_level",
        "quality_source_requests", "quality_source_data",
        "quality_p2p_requests", "quality_p2p_data", "quality_data_mb",
        "total_source_data", "total_p2p_data", "total_data_mb",
        "eventDate"
    ]
    
    df = spark.table(TABLES["silver"])
    actual_columns = df.columns
    
    missing = [col for col in expected_columns if col not in actual_columns]
    
    if missing:
        return False, f"Missing columns: {missing}", False
    
    # Verify no nested structures remain
    schema = df.schema
    complex_types = [f.name for f in schema.fields if f.dataType.typeName() in ["struct", "map", "array"]]
    
    if complex_types:
        return False, f"Unexpected nested structures: {complex_types}", False
    
    return True, f"Silver schema valid with {len(actual_columns)} columns (fully flattened)", False

run_test("Silver Layer Schema Validation", test_silver_schema, "Schema Validation")

In [0]:
# Test 4: Validate NOT NULL constraints in silver layer
def test_not_null_constraints():
    """
    Verify required columns have no null values in silver layer.
    """
    # Columns that should never be null based on schema definition
    required_columns = [
        "customerId", "contentId", "clientId",
        "timestamp_server", "buffering_count", "buffering_time_ms",
        "buffering_time_sec", "quality_level",
        "total_source_data", "total_p2p_data", "total_data_mb",
        "eventDate"
    ]
    
    df = spark.table(TABLES["silver"])
    total_rows = df.count()
    
    if total_rows == 0:
        return True, "No data to validate (0 rows in silver table)", True
    
    null_violations = []
    
    for col in required_columns:
        null_count = df.filter(F.col(col).isNull()).count()
        if null_count > 0:
            null_violations.append(f"{col}: {null_count} nulls")
            print(f"  ✗ {col} has {null_count} null values ({null_count/total_rows*100:.2f}%)")
        else:
            print(f"  ✓ {col} has no nulls")
    
    if null_violations:
        return False, f"NOT NULL violations found: {'; '.join(null_violations)}", False
    else:
        return True, f"All {len(required_columns)} required columns have no nulls ({total_rows:,} rows checked)", False

run_test("NOT NULL Constraints Validation", test_not_null_constraints, "Data Quality")

In [0]:
# Test 5: Validate data value ranges
def test_value_ranges():
    """
    Verify data values are within expected ranges (matching expectations in pipeline).
    """
    df = spark.table(TABLES["silver"])
    total_rows = df.count()
    
    if total_rows == 0:
        return True, "No data to validate (0 rows in silver table)", True
    
    violations = []
    
    # Test: buffering_count >= 0
    negative_buffering = df.filter(F.col("buffering_count") < 0).count()
    if negative_buffering > 0:
        violations.append(f"negative buffering_count: {negative_buffering} rows")
        print(f"  ✗ Found {negative_buffering} rows with negative buffering_count")
    else:
        print(f"  ✓ buffering_count >= 0 for all rows")
    
    # Test: buffering_time_ms >= 0
    negative_time = df.filter(F.col("buffering_time_ms") < 0).count()
    if negative_time > 0:
        violations.append(f"negative buffering_time_ms: {negative_time} rows")
        print(f"  ✗ Found {negative_time} rows with negative buffering_time_ms")
    else:
        print(f"  ✓ buffering_time_ms >= 0 for all rows")
    
    # Test: total_source_data >= 0 AND total_p2p_data >= 0
    negative_data = df.filter(
        (F.col("total_source_data") < 0) | (F.col("total_p2p_data") < 0)
    ).count()
    if negative_data > 0:
        violations.append(f"negative data volumes: {negative_data} rows")
        print(f"  ✗ Found {negative_data} rows with negative data volumes")
    else:
        print(f"  ✓ total_source_data and total_p2p_data >= 0 for all rows")
    
    # Test: quality_level IS NOT NULL
    null_quality = df.filter(F.col("quality_level").isNull()).count()
    if null_quality > 0:
        violations.append(f"null quality_level: {null_quality} rows")
        print(f"  ✗ Found {null_quality} rows with null quality_level")
    else:
        print(f"  ✓ quality_level is not null for all rows")
    
    # Warning: Check for unreasonable buffering times (>1 hour = 3600 seconds)
    excessive_buffering = df.filter(F.col("buffering_time_sec") > 3600).count()
    if excessive_buffering > 0:
        pct = excessive_buffering / total_rows * 100
        print(f"  ⚠ Warning: {excessive_buffering} rows ({pct:.2f}%) have buffering > 1 hour")
        # This is a warning, not a failure
    
    if violations:
        return False, f"Value range violations: {'; '.join(violations)}", False
    elif excessive_buffering > 0:
        return True, f"All range checks passed with warnings ({total_rows:,} rows checked)", True
    else:
        return True, f"All value ranges valid ({total_rows:,} rows checked)", False

run_test("Value Range Validation", test_value_ranges, "Data Quality")

In [0]:
# Test 6: Verify data flows from bronze to silver
def test_bronze_to_silver_flow():
    """
    Verify that silver layer contains data derived from bronze layer.
    Silver row count should be >= bronze (due to quality distribution explosion).
    """
    bronze_df = spark.table(TABLES["bronze"])
    silver_df = spark.table(TABLES["silver"])
    
    bronze_count = bronze_df.count()
    silver_count = silver_df.count()
    
    if bronze_count == 0:
        return True, "No bronze data to validate data flow", True
    
    print(f"  Bronze rows: {bronze_count:,}")
    print(f"  Silver rows: {silver_count:,}")
    
    # Silver should have MORE rows due to quality distribution explosion
    # Each bronze row with N quality levels becomes N silver rows
    if silver_count < bronze_count:
        return False, f"Silver has fewer rows than bronze ({silver_count} < {bronze_count}). Expected explosion due to quality distribution.", False
    
    # Check that some customers exist in both
    bronze_customers = set(row[0] for row in bronze_df.select("customerId").distinct().collect())
    silver_customers = set(row[0] for row in silver_df.select("customerId").distinct().collect())
    
    common_customers = bronze_customers.intersection(silver_customers)
    print(f"  Common customers: {len(common_customers)}")
    
    if len(common_customers) == 0 and bronze_count > 0:
        return False, "No customers found in both bronze and silver layers", False
    
    explosion_ratio = silver_count / bronze_count if bronze_count > 0 else 0
    print(f"  Explosion ratio: {explosion_ratio:.2f}x (due to quality distribution)")
    
    return True, f"Data flows bronze→silver: {bronze_count:,} → {silver_count:,} rows ({explosion_ratio:.2f}x explosion)", False

run_test("Bronze to Silver Data Flow", test_bronze_to_silver_flow, "Data Flow")

In [0]:
# Test 7: Validate quarantine logic captures bad data
def test_quarantine_logic():
    """
    Verify that quarantine table captures records that fail quality checks.
    """
    quarantine_df = spark.table(TABLES["quarantine"])
    quarantine_count = quarantine_df.count()
    
    print(f"  Quarantine rows: {quarantine_count:,}")
    
    if quarantine_count == 0:
        bronze_count = spark.table(TABLES["bronze"]).count()
        if bronze_count == 0:
            return True, "No quarantine data (no bronze data ingested yet)", True
        else:
            return True, "No quarantine data (100% data quality - excellent!)", False
    
    # Verify quarantine table has required columns
    expected_cols = ["quarantine_reason", "quarantine_timestamp", "customerId"]
    actual_cols = quarantine_df.columns
    missing = [col for col in expected_cols if col not in actual_cols]
    
    if missing:
        return False, f"Quarantine table missing columns: {missing}", False
    
    # Check quarantine reasons distribution
    print("\n  Quarantine reasons:")
    reasons = quarantine_df.groupBy("quarantine_reason").count().orderBy(F.desc("count")).collect()
    for row in reasons[:5]:  # Top 5 reasons
        print(f"    - {row['quarantine_reason']}: {row['count']} records")
    
    return True, f"Quarantine logic working: {quarantine_count:,} records quarantined with {len(reasons)} distinct reasons", False

run_test("Quarantine Logic Validation", test_quarantine_logic, "Data Quality")

In [0]:
# Test 8: Validate gold layer metric calculations
def test_gold_aggregations():
    """
    Verify gold layer contains aggregated metrics.
    """
    try:
        gold_df = spark.table(TABLES["gold"])
        gold_count = gold_df.count()
        
        print(f"  Gold rows: {gold_count:,}")
        
        if gold_count == 0:
            silver_count = spark.table(TABLES["silver"]).count()
            if silver_count == 0:
                return True, "No gold data (no silver data to aggregate yet)", True
            else:
                return False, f"No gold data despite {silver_count:,} silver rows", False
        
        # Verify gold has aggregation columns
        expected_agg_cols = ["window_start", "window_end", "customerId", "clientId"]
        actual_cols = gold_df.columns
        missing = [col for col in expected_agg_cols if col not in actual_cols]
        
        if missing:
            print(f"  ⚠ Gold table missing expected columns: {missing}")
        
        # Check that gold has fewer rows than silver (aggregation happened)
        silver_count = spark.table(TABLES["silver"]).count()
        print(f"  Silver rows: {silver_count:,}")
        print(f"  Aggregation ratio: {silver_count / gold_count if gold_count > 0 else 0:.2f}x reduction")
        
        if gold_count >= silver_count and silver_count > 0:
            return False, f"Gold has more/equal rows than silver ({gold_count} >= {silver_count}). Expected aggregation/reduction.", False
        
        return True, f"Gold layer aggregations working: {silver_count:,} → {gold_count:,} rows", False
        
    except Exception as e:
        return False, f"Error accessing gold table: {str(e)}", False

run_test("Gold Layer Aggregation Validation", test_gold_aggregations, "Data Flow")

In [0]:
# Test 9: Validate data freshness
def test_data_freshness():
    """
    Verify that the most recent data is not too old.
    """
    bronze_df = spark.table(TABLES["bronze"])
    
    if bronze_df.count() == 0:
        return True, "No data to check freshness", True
    
    # Get latest timestamp from bronze
    latest_timestamp = bronze_df.selectExpr("MAX(timestampInfo.server) as max_ts").collect()[0]["max_ts"]
    
    if latest_timestamp is None:
        return False, "Latest timestamp is NULL in bronze layer", False
    
    # Convert to datetime
    latest_dt = datetime.datetime.fromtimestamp(latest_timestamp / 1000)
    age_hours = (datetime.datetime.now() - latest_dt).total_seconds() / 3600
    
    print(f"  Latest event: {latest_dt}")
    print(f"  Age: {age_hours:.2f} hours")
    
    # Warning if data is older than 24 hours
    if age_hours > 24:
        return True, f"Data is {age_hours:.1f} hours old (consider refreshing)", True
    elif age_hours > 2:
        return True, f"Data is {age_hours:.1f} hours old", False
    else:
        return True, f"Data is fresh ({age_hours:.1f} hours old)", False

run_test("Data Freshness Check", test_data_freshness, "Pipeline Health")

In [0]:
# Test 10: Validate query performance
def test_query_performance():
    """
    Verify that basic queries execute within acceptable time limits.
    """
    import time
    
    # Test 1: Simple count query on silver
    start = time.time()
    count = spark.table(TABLES["silver"]).count()
    elapsed = time.time() - start
    
    print(f"  Silver count query: {elapsed:.2f}s ({count:,} rows)")
    
    if elapsed > 30:
        return False, f"Count query took {elapsed:.2f}s (>30s threshold)", False
    elif elapsed > 10:
        return True, f"Count query took {elapsed:.2f}s (acceptable but could be optimized)", True
    else:
        return True, f"Count query took {elapsed:.2f}s (good performance)", False

run_test("Query Performance Check", test_query_performance, "Performance")

In [0]:
# Test 11: Validate pipeline configuration
def test_pipeline_configuration():
    """
    Verify pipeline is configured correctly for production.
    """
    issues = []
    
    # Check if pipeline exists and get config
    try:
        pipeline_info = spark.sql(f"""
            SELECT * FROM system.lakeflow.pipelines 
            WHERE pipeline_id = '{PIPELINE_ID}'
        """).collect()
        
        if len(pipeline_info) == 0:
            return False, f"Pipeline {PIPELINE_ID} not found in system.lakeflow.pipelines", False
        
        pipeline = pipeline_info[0]
        print(f"  Pipeline name: {pipeline.get('name', 'N/A')}")
        print(f"  Pipeline state: {pipeline.get('state', 'N/A')}")
        
        # Check Photon is enabled
        if 'photon' in pipeline:
            photon_enabled = pipeline['photon']
            if photon_enabled:
                print(f"  ✓ Photon: enabled")
            else:
                issues.append("Photon not enabled")
                print(f"  ✗ Photon: disabled")
        
        # Check serverless
        if 'serverless' in pipeline:
            serverless = pipeline['serverless']
            if serverless:
                print(f"  ✓ Serverless: enabled")
            else:
                print(f"  ⚠ Serverless: disabled (consider enabling for auto-scaling)")
        
    except Exception as e:
        return False, f"Error checking pipeline config: {str(e)}", False
    
    if issues:
        return False, f"Configuration issues: {'; '.join(issues)}", False
    else:
        return True, "Pipeline configuration looks good", False

run_test("Pipeline Configuration Check", test_pipeline_configuration, "Pipeline Health")

## 📊 Test Results Summary

Run the cell below to see the complete test results summary.

In [0]:
# Display test results summary
import pandas as pd

if not test_results:
    print("⚠️ No tests have been run yet. Run all cells above to execute the test suite.")
else:
    # Create summary dataframe
    summary_df = pd.DataFrame(test_results)
    
    # Count by status
    status_counts = summary_df['status'].value_counts()
    
    total_tests = len(test_results)
    passed = status_counts.get('✅ PASS', 0)
    failed = status_counts.get('❌ FAIL', 0)
    warnings = status_counts.get('⚠️ WARN', 0)
    
    print("\n" + "="*80)
    print("TEST SUITE SUMMARY")
    print("="*80)
    print(f"\nTotal Tests: {total_tests}")
    print(f"  ✅ Passed: {passed}")
    print(f"  ❌ Failed: {failed}")
    print(f"  ⚠️ Warnings: {warnings}")
    print(f"\nSuccess Rate: {passed/total_tests*100:.1f}%")
    
    # Show failed tests
    if failed > 0:
        print("\n" + "="*80)
        print("FAILED TESTS - REQUIRES ATTENTION")
        print("="*80)
        failed_tests = summary_df[summary_df['status'] == '❌ FAIL']
        for idx, row in failed_tests.iterrows():
            print(f"\n❌ {row['test_name']}")
            print(f"   Category: {row['category']}")
            print(f"   Message: {row['message']}")
    
    # Show warnings
    if warnings > 0:
        print("\n" + "="*80)
        print("WARNINGS - REVIEW RECOMMENDED")
        print("="*80)
        warning_tests = summary_df[summary_df['status'] == '⚠️ WARN']
        for idx, row in warning_tests.iterrows():
            print(f"\n⚠️ {row['test_name']}")
            print(f"   Category: {row['category']}")
            print(f"   Message: {row['message']}")
    
    # Show summary by category
    print("\n" + "="*80)
    print("RESULTS BY CATEGORY")
    print("="*80)
    category_summary = summary_df.groupby('category')['status'].value_counts().unstack(fill_value=0)
    print(category_summary)
    
    # Overall assessment
    print("\n" + "="*80)
    if failed == 0 and warnings == 0:
        print("🎉 ALL TESTS PASSED - PIPELINE IS PRODUCTION READY!")
    elif failed == 0:
        print("✅ ALL TESTS PASSED (with warnings) - Review warnings before production deployment")
    else:
        print("⚠️ SOME TESTS FAILED - Fix failed tests before production deployment")
    print("="*80 + "\n")
    
    # Display detailed results table
    display(summary_df[['category', 'test_name', 'status', 'message']])

## 🚀 Next Steps

Based on test results:

### If All Tests Pass:
1. ✅ Schedule this testing notebook to run weekly
2. ✅ Configure alerts for test failures
3. ✅ Document test results in your pipeline runbook
4. ✅ Proceed with production deployment

### If Tests Fail:
1. ❌ Review failed test messages above
2. ❌ Fix issues in pipeline code or configuration
3. ❌ Re-run tests after fixes
4. ❌ Do not deploy to production until all tests pass

### Recommended Schedule:
- **Development**: Run tests after every pipeline code change
- **Staging**: Run tests daily (automated)
- **Production**: Run tests weekly (automated) + after any deployment

## 📋 How to Schedule This Testing Suite

1. Click the "Schedule" button in the notebook toolbar
2. Configure job settings:
   - **Name**: `Video Telemetry Pipeline - Weekly Tests`
   - **Schedule**: Weekly (Sunday at 2 AM)
   - **Compute**: Use pipeline's compute or serverless
   - **Notifications**: Enable email on failure
3. Save and enable the job
4. Create an alert for job failures

## 🔍 Integration with CI/CD

For Declarative Automation Bundles (DABs):
```yaml
resources:
  jobs:
    pipeline_test_suite:
      name: "Pipeline Testing Suite"
      tasks:
        - task_key: run_tests
          notebook_task:
            notebook_path: "operations/Pipeline_Testing_Suite"
          existing_cluster_id: "${var.cluster_id}"
      schedule:
        quartz_cron_expression: "0 0 2 ? * SUN"
        timezone_id: "UTC"
      email_notifications:
        on_failure:
          - your-team@company.com
```